# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
import numpy as np
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    all_move_probabilities,
    single_move_probability,
    stabilizer_edges,
    stab_labels,
    move_error
)
import scipy
import timeit
import jax
import jax.numpy as jnp
import numpy as np
from typing import Tuple
from jax import Array
from jax.typing import ArrayLike

In [2]:
length = 5
width = 5
p = 3
d = 2 * p
moebius_code = MoebiusCodeTwoOddPrime(length=length, width=width, d=d)
h_z = moebius_code.h_z
h_x = moebius_code.h_x
h_x_mod_2 = moebius_code.h_x_mod_2
h_x_mod_p = moebius_code.h_x_mod_p
logical_x = moebius_code.logical_x
logical_z = moebius_code.logical_z
gamma_t = 0.1
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=d, gamma_t=gamma_t
)

base_key = jax.random.PRNGKey(42)
initial_error = em_lindblad.generate_random_error(base_key)
initial_error_mod_2 = jnp.mod(initial_error, 2)
initial_error_mod_p = jnp.mod(initial_error, p)
# Here we consider the full syndrome including the plaquette
# we usually remove because of the constraint as this simplified the
# coding of the worm algorithm. In fact, in this the syndromes will
# always be annihilated in pairs, and the total number of syndromes is
# always even as one can check numerically.
syndrome = moebius_code.get_plaquette_syndrome(initial_error)
syndrome_mod_2 = jnp.mod(syndrome, 2)
syndrome_mod_p = jnp.mod(syndrome, p)
num_plaquette, num_edges = h_x.shape

In [3]:
def run_worm(
    base_key: Array,
    initial_error_mod_2: Array,
    initial_error_mod_p: Array,
    h_x_mod_2: Array,
    h_x_mod_p: Array,
    num_plaquette: int,
    num_edges: int,
    error_model: ErrorModelLindbladTwoOddPrime,
    max_steps: int,
) -> Array:

    def random_edge_boundary(key):
        return jax.random.randint(key, 1, 0, 3)

    def random_edge_bulk(key):
        return jax.random.randint(key, 1, 0, 4)

    def initialize_worm(key):
        """In the first iteration I will initialize a fixed error"""
        pass

    def worm_step(carry_worm_step, x):
        (worm_error_mod_2, worm_error_mod_p, worm_syndrome_mod_2,
            worm_syndrome_mod_p, head, tail, continue_worm, key) = carry_worm_step
        # Split the key into a new key that will get passed at
        # the end and a subkey that will be used in this step
        key, subkey = jax.random.split(key)

        def reject(args):
            pass

        def accept(args):
            pass

Let us break the problem into sub-problems. Let us define a function that given an edge, returns the 
plaquette stabilizers $\mod p$ associated with the two plaquettes that touch it.

Now we want a function that given, a plaquette $\mod p$ returns the probabilities of having an error $\mod 2$ at one of its edges, and all the powers $0, 1, ..., p - 1$ of the plaquette itself. I note that I believe that it is important to choose the moves weighted by these probabilities. In particular, even the random choice of head move, should be weighted, because the plaquette of weight $3$ at the boundary would have a different probability of those of weight $4$ in the bulk. 

While the previous might be useful I think that what we really want is given a head, i.e., a plaquette index to obtain the probability of each possible move. How many are the possible moves? If the plaquette is weight $4$ these are $4 p$, if weight $3$ they are $3 p$. We can then represent a "move" by an edge, i.e., its label, and a power between $0, 1, \dots, p-1$ which represents the power of the corresponding odd stabilizer. So let us write a function that given a head label, gives back its edges. 

In [4]:
head = 0
error_mod_2 = jnp.zeros(moebius_code.num_edges, dtype=jnp.int16)
error_mod_p = jnp.zeros(moebius_code.num_edges, dtype=jnp.int16)
func = jax.jit(all_move_probabilities, static_argnums=(4, 5))
func(error_mod_2, error_mod_p, head, h_x_mod_p, p, em_lindblad)

(Array([[ 1.5471634e-04,  5.7153695e-09,  5.7156906e-09],
        [ 1.8709533e-04,  1.3904039e-06,  1.3904064e-06],
        [ 1.8709533e-04,  1.3904064e-06,  1.3904039e-06],
        [-1.0000000e+00, -1.0000000e+00, -1.0000000e+00]], dtype=float32),
 Array([ 0, 20, 21, -1], dtype=int32))

In [7]:
head = 6
head_edges = stabilizer_edges(head, h_x_mod_p)
edge = head_edges[0]
power = 2
jit_move_error = jax.jit(move_error)
my_move = jit_move_error(edge, power, head, h_x_mod_p, p)
print(edge)
print(my_move)

1
[[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 2 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0]]


In [16]:
incident_stabs = stab_labels(edge, h_x_mod_p)
head_is_first = incident_stabs[0] == head

candidate_head = jax.lax.cond(
    head_is_first,
    lambda: incident_stabs[1],
    lambda: incident_stabs[0]
)

candidate_stab = h_x_mod_p[candidate_head]
print(jnp.mod(power * candidate_stab, p) - my_move[1, :])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [21]:
def fun_example(
    x: ArrayLike,
) -> ArrayLike:
    return x[0, :]


jit_fun_example = jax.jit(fun_example)
pippo = jnp.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
jit_fun_example(pippo)

Array([1., 2., 3.], dtype=float32)

In [17]:
pippo = 0 * jnp.array([1.0, 2.0, 3.0])
pippo.at[0].set(1.0)

Array([1., 0., 0.], dtype=float32)

In [5]:
head_edges = stabilizer_edges(head, h_x_mod_p)
test_edge = head_edges[1]
incident_plaquettes = stab_labels(test_edge, h_x_mod_p)
head_is_first = incident_plaquettes[0] == head

candidate_head = jax.lax.cond(
    head_is_first,
    lambda: incident_plaquettes[1],
    lambda: incident_plaquettes[0]
)
print(candidate_head)
candidate_head_edges = stabilizer_edges(candidate_head, h_x_mod_p)
print(candidate_head_edges)
candidate_head_stab = jnp.mod(0 * h_x_mod_p[candidate_head, :], p)

prob = 1.0
for edge in candidate_head_edges:
    print(edge)
    if edge != -1:
        if edge == test_edge:
            prob = prob * \
                em_lindblad.get_modular_probability(
                    1, candidate_head_stab[edge])
        else:
            prob = prob * \
                em_lindblad.get_modular_probability(
                    0, candidate_head_stab[edge])

print(prob)

4
[19 20 44 -1]
19
20
44
-1
0.00018709533


In [6]:
func(error_mod_2, error_mod_p, 10, h_x_mod_p, p, em_lindblad)

(Array([[1.5471634e-04, 5.7153695e-09, 5.7156906e-09],
        [1.5471634e-04, 5.7153695e-09, 5.7156906e-09],
        [1.5471634e-04, 5.7153695e-09, 5.7156906e-09],
        [1.5471634e-04, 5.7156906e-09, 5.7153695e-09]], dtype=float32),
 Array([ 5, 10, 30, 31], dtype=int32))